In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier
from sklearn.metrics import precision_recall_curve


In [168]:
df_raw = pd.read_csv("../data/cs-training.csv", index_col=0)
print(df_raw.shape)
df_raw.info(memory_usage="deep")
df_raw.head()

(150000, 11)
<class 'pandas.DataFrame'>
RangeIndex: 150000 entries, 1 to 150000
Data columns (total 11 columns):
 #   Column                                Non-Null Count   Dtype  
---  ------                                --------------   -----  
 0   SeriousDlqin2yrs                      150000 non-null  int64  
 1   RevolvingUtilizationOfUnsecuredLines  150000 non-null  float64
 2   age                                   150000 non-null  int64  
 3   NumberOfTime30-59DaysPastDueNotWorse  150000 non-null  int64  
 4   DebtRatio                             150000 non-null  float64
 5   MonthlyIncome                         120269 non-null  float64
 6   NumberOfOpenCreditLinesAndLoans       150000 non-null  int64  
 7   NumberOfTimes90DaysLate               150000 non-null  int64  
 8   NumberRealEstateLoansOrLines          150000 non-null  int64  
 9   NumberOfTime60-89DaysPastDueNotWorse  150000 non-null  int64  
 10  NumberOfDependents                    146076 non-null  float64
dty

,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
1,1,0.766127,45,2,0.802982,9120.0,13,0,6,0,2.0
2,0,0.957151,40,0,0.121876,2600.0,4,0,0,0,1.0
3,0,0.658180,38,1,0.085113,3042.0,2,1,0,0,0.0
4,0,0.233810,30,0,0.036050,3300.0,5,0,0,0,0.0
5,0,0.907239,49,1,0.024926,63588.0,7,0,1,0,0.0


## Анализ данных

In [169]:
duplicates_cnt = df_raw.duplicated().sum()
print(f"duplicates: {duplicates_cnt} ({duplicates_cnt / len(df_raw):.2%})")
df = df_raw.drop_duplicates().reset_index(drop=True)

duplicates: 609 (0.41%)


In [170]:
missing = pd.DataFrame({"miss_cnt": df.isna().sum(), "miss_pct": df.isna().mean() * 100}).query("miss_cnt > 0")
print(missing)

                    miss_cnt   miss_pct
MonthlyIncome          29221  19.560081
NumberOfDependents      3828   2.562403


In [171]:
df["SeriousDlqin2yrs"].value_counts(normalize=True)

SeriousDlqin2yrs
0    0.933001
1    0.066999
Name: proportion, dtype: float64

In [172]:
print(df["NumberOfDependents"].value_counts(dropna=False, normalize=True))

NumberOfDependents
0.0     0.578295
1.0     0.176142
2.0     0.130671
3.0     0.063478
NaN     0.025624
4.0     0.019158
5.0     0.004994
6.0     0.001058
7.0     0.000341
8.0     0.000161
10.0    0.000033
9.0     0.000033
20.0    0.000007
13.0    0.000007
Name: proportion, dtype: float64


NumberOfDependents: пропущено 2,5%. Можно предположить, что пустая ячейка означает отсутствие иждивенцев. Около 60% - 0. После сплита заполним модой трейна

MonthlyIncome: пропущено 19,5%. Неуказанный доход может быть важным фактором, замазывать медианой или модой будет грубо. Оставим пропуски, но создадим бинарный флаг при фильтрации данных.

In [173]:
num_stats = df.describe(percentiles=[0.01, 0.05, 0.50, 0.75, 0.95, 0.99]).T[["min", "1%", "5%", "50%", "75%", "95%", "99%", "max"]]
num_stats

,min,1%,5%,50%,75%,95%,99%,max
SeriousDlqin2yrs,0.0,0.0,0.000000,0.000000,0.000000,1.0,1.000000,1.0
RevolvingUtilizationOfUnsecuredLines,0.0,0.0,0.000000,0.154235,0.556494,1.0,1.093922,50708.0
age,0.0,24.0,29.000000,52.000000,63.000000,78.0,87.000000,109.0
NumberOfTime30-59DaysPastDueNotWorse,0.0,0.0,0.000000,0.000000,0.000000,2.0,4.000000,98.0
DebtRatio,0.0,0.0,0.005199,0.368234,0.875279,2455.5,4985.100000,329664.0
MonthlyIncome,0.0,0.0,1300.000000,5400.000000,8250.000000,14594.1,25000.000000,3008750.0
NumberOfOpenCreditLinesAndLoans,0.0,0.0,2.000000,8.000000,11.000000,18.0,24.000000,58.0
NumberOfTimes90DaysLate,0.0,0.0,0.000000,0.000000,0.000000,1.0,3.000000,98.0
NumberRealEstateLoansOrLines,0.0,0.0,0.000000,1.000000,2.000000,3.0,4.000000,54.0
NumberOfTime60-89DaysPastDueNotWorse,0.0,0.0,0.000000,0.000000,0.000000,1.0,2.000000,98.0


In [174]:
print((df["age"] < 18).sum())

1


Очевидная аномалия, удалим на этапе фильтрации данных

In [175]:
print((df["NumberOfTime30-59DaysPastDueNotWorse"] >= 96).sum(), ((df["NumberOfTime30-59DaysPastDueNotWorse"] >= 96).mean() * 100).round(3))
print((df["NumberOfTimes90DaysLate"] >= 96).sum())
print((df["NumberOfTime60-89DaysPastDueNotWorse"] >= 96).sum())
cols = ["NumberOfTime30-59DaysPastDueNotWorse", "NumberOfTimes90DaysLate", "NumberOfTime60-89DaysPastDueNotWorse"]
print((df[cols] >= 96).all(axis=1).sum())
df[(df[cols] >= 96).all(axis=1)][cols].value_counts()

225 0.151
225
225
225


NumberOfTime30-59DaysPastDueNotWorse  NumberOfTimes90DaysLate  NumberOfTime60-89DaysPastDueNotWorse
98                                    98                       98                                      220
96                                    96                       96                                        5
Name: count, dtype: int64

Аномалия. Причем можно подметить, что в случае аномалии, все 3 фичи имеют одинаковое значение. в двльнейшем создадим бинарный флаг для таких случаев.

In [176]:
print((df["RevolvingUtilizationOfUnsecuredLines"] > 2).sum(), (df["RevolvingUtilizationOfUnsecuredLines"] > 2).mean() * 100)

371 0.24834160023026824


Будем считать, что 2.0 (200%) утилизации - это глубокий овердрафт. Больше 200% всего 0.24% строк и скорее всего это технические выбросы, в будущем ограничим сверху до 2.0

In [177]:
print((df["DebtRatio"] > 1).sum())

35130


Очевидно, что если поле Дохода = NaN, то DebtRatio отображает долг клиента. Если доход есть, то скорее всего он находится в диапазоне от 0 до 1.

In [178]:
df[(df["DebtRatio"] > 100) & (df["MonthlyIncome"] !=None)]["MonthlyIncome"].value_counts()

MonthlyIncome
0.0      1219
1.0       455
2.0         5
10.0        2
21.0        1
25.0        1
9.0         1
5.0         1
15.0        1
401.0       1
Name: count, dtype: int64

В дальнейшем применим логарифмирование, чтобы сжать тяжелый правый хвост распределения.

## Фильтрация данных

In [179]:
df["MonthlyIncome_isna"] = df["MonthlyIncome"].isna().astype(int)
df.groupby("MonthlyIncome_isna")["SeriousDlqin2yrs"].agg(["count", "mean"])
df.groupby("MonthlyIncome_isna")["DebtRatio"].describe(percentiles=[0.01, 0.05, 0.1, 0.5, 0.75, 0.95, 0.99])

,count,mean,std,min,1%,5%,10%,50%,75%,95%,99%,max
MonthlyIncome_isna,,,,,,,,,,,,
0,120170.0,26.620680,424.620573,0.0,0.0,0.004665,0.022565,0.296174,0.482709,1.129511,662.965,61106.5
1,29221.0,1702.565005,4279.494336,0.0,0.0,2.000000,15.000000,1198.000000,2407.000000,4930.000000,8139.000,329664.0


In [180]:
df["overdue_anomaly"] = (df["NumberOfTimes90DaysLate"] >= 96).astype(int)
df.groupby("overdue_anomaly")["SeriousDlqin2yrs"].agg(["count", "mean"])

,count,mean
overdue_anomaly,,
0,149166,0.066188
1,225,0.604444


60% строк с такой аномалией вышли в дефолт. В 10 раз больше базового процента. Важный признак

In [181]:
df_clean = df[df["age"] > 0].copy()

df_clean["RevolvingUtilizationOfUnsecuredLines"] = df_clean["RevolvingUtilizationOfUnsecuredLines"].clip(upper=2)

df_clean["DebtRatio"] = np.log1p(df_clean["DebtRatio"])

In [182]:
X = df_clean.drop(columns=["SeriousDlqin2yrs"])
y = df_clean["SeriousDlqin2yrs"]

X_train, X_val_test, y_train, y_val_test = train_test_split(X, y, random_state=67, test_size=0.3, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_val_test, y_val_test, random_state=67, test_size=0.5, stratify=y_val_test)

mInc = X_train["MonthlyIncome"].median()
mDep = X_train["NumberOfDependents"].median()

# Baseline: LofisticRegression

In [183]:
scaler = StandardScaler()
scaler.set_output(transform="pandas")
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled["MonthlyIncome"] = X_train["MonthlyIncome"].fillna(mInc)
X_test_scaled["MonthlyIncome"] = X_test["MonthlyIncome"].fillna(mInc)

X_train_scaled["NumberOfDependents"] = X_train["NumberOfDependents"].fillna(mDep)
X_test_scaled["NumberOfDependents"] = X_test["NumberOfDependents"].fillna(mDep)



X_train_scaled = scaler.fit_transform(X_train_scaled)
X_test_scaled = scaler.transform(X_test_scaled)


In [184]:
logreg = LogisticRegression()
logreg.fit(X_train_scaled, y_train)

proba = logreg.predict_proba(X_test_scaled)[:,1]

In [185]:
roc_auc = roc_auc_score(y_test, proba)
gini = 2*roc_auc - 1

print(f"ROC AUC:{roc_auc}, GINI:{gini * 100}")

ROC AUC:0.853929713587349, GINI:70.78594271746981


In [186]:
catboost = CatBoostClassifier(iterations=500, learning_rate=0.05, eval_metric="AUC", random_seed=67, verbose=100)
catboost.fit(X_train, y_train, eval_set=(X_val, y_val), early_stopping_rounds=50)
probabilities = catboost.predict_proba(X_test)[:,1]

0:	test: 0.7933515	best: 0.7933515 (0)	total: 189ms	remaining: 1m 34s
100:	test: 0.8611606	best: 0.8611711 (98)	total: 3.04s	remaining: 12s
200:	test: 0.8628599	best: 0.8628599 (200)	total: 5.71s	remaining: 8.49s
300:	test: 0.8636325	best: 0.8636687 (284)	total: 8.44s	remaining: 5.58s
400:	test: 0.8639604	best: 0.8639604 (400)	total: 11.1s	remaining: 2.73s
499:	test: 0.8640154	best: 0.8640392 (449)	total: 13.7s	remaining: 0us

bestTest = 0.8640391905
bestIteration = 449

Shrink model to first 450 iterations.


In [187]:
roc_auc = roc_auc_score(y_test, probabilities)
gini = 2*roc_auc - 1

print(f"ROC AUC:{roc_auc}, GINI:{gini * 100}")

ROC AUC:0.8704446430285361, GINI:74.08892860570721


In [189]:
importance = pd.Series(catboost.get_feature_importance(), index=X_train.columns).sort_values(ascending=False)
print(importance)

RevolvingUtilizationOfUnsecuredLines    32.657001
NumberOfTime30-59DaysPastDueNotWorse    14.438953
age                                     10.645259
NumberOfTimes90DaysLate                 10.170949
DebtRatio                                8.388709
NumberOfTime60-89DaysPastDueNotWorse     6.402240
NumberOfOpenCreditLinesAndLoans          6.021062
MonthlyIncome                            5.151618
NumberRealEstateLoansOrLines             3.745690
NumberOfDependents                       2.170777
MonthlyIncome_isna                       0.207742
overdue_anomaly                          0.000000
dtype: float64


In [188]:
from sklearn.metrics import precision_recall_curve

precisions, recalls, thresholds = precision_recall_curve(y_test, probabilities)
f1_scores = 2 * (precisions[:-1] * recalls[:-1]) / (precisions[:-1] + recalls[:-1] + 1e-8)

best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]
print(f"Оптимальный порог: {best_threshold:.4f}")
print(f"Лучший F1-Score: {f1_scores[best_idx]:.4f}")

Оптимальный порог: 0.1707
Лучший F1-Score: 0.4526
